<a href="https://colab.research.google.com/github/andreysenna/projeto_sql_vendas/blob/main/Projeto_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Criação do Notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/projeto_sql_vendas/dados"
!mkdir -p "content/drive/MyDrive/projeto_sql_vendas/resultados"

print('Pasta criada.')

Mounted at /content/drive
Pasta criada.


# 2. Bibliotecas que serão utilizadas e criação do banco de dados.

In [ ]:
import sqlite3
import pandas as pd

db = sqlite3.connect(':memory:')
print('Banco de dados criado com sucesso.')

Banco de dados criado com sucesso.


# 3. Criação de tabelas.

In [ ]:
db.execute('''
CREATE TABLE vendas (
  id INTEGER PRIMARY KEY,
  data TEXT,
  produto TEXT,
  categoria TEXT,
  quantidade INTEGER,
  preco_unitario REAL,
  regiao TEXT
)
''')

# Agora, irei inserir dados nesta tabela (que foram gerados por IA para agilizar
# esse processo.)

vendas_data = [
    ('2025-01-15', 'Camiseta', 'Vestuário', 5, 29.90, 'Sudeste'),
    ('2025-01-20', 'Calça', 'Vestuário', 3, 89.90, 'Sul'),
    ('2025-02-05', 'Tênis', 'Calçados', 2, 199.90, 'Sudeste'),
    ('2025-02-18', 'Boné', 'Acessórios', 10, 39.90, 'Nordeste'),
    ('2025-03-10', 'Camiseta', 'Vestuário', 8, 29.90, 'Sudeste'),
    ('2025-03-22', 'Calça', 'Vestuário', 4, 89.90, 'Centro-Oeste'),
    ('2025-04-02', 'Tênis', 'Calçados', 3, 199.90, 'Sul'),
    ('2025-04-14', 'Meia', 'Acessórios', 15, 19.90, 'Nordeste'),
    ('2025-05-01', 'Camiseta', 'Vestuário', 12, 29.90, 'Sudeste'),
    ('2025-05-20', 'Boné', 'Acessórios', 6, 39.90, 'Sul'),
    ('2025-06-10', 'Calça', 'Vestuário', 2, 89.90, 'Nordeste'),
    ('2025-06-25', 'Tênis', 'Calçados', 4, 199.90, 'Sudeste'),
]

db.executemany('''
INSERT INTO vendas (data, produto, categoria, quantidade,
preco_unitario, regiao)
VALUES (?, ?, ?, ?, ?, ?)
''', vendas_data)
db.commit()

#Agora, irei verificar se está tudo correto.

df = pd.read_sql_query('SELECT * FROM vendas', db)
print(f"✅ {len(df)} registros inseridos")
df.head()

✅ 12 registros inseridos


,id,data,produto,categoria,quantidade,preco_unitario,regiao
0,1,2025-01-15,Camiseta,Vestuário,5,29.9,Sudeste
1,2,2025-01-20,Calça,Vestuário,3,89.9,Sul
2,3,2025-02-05,Tênis,Calçados,2,199.9,Sudeste
3,4,2025-02-18,Boné,Acessórios,10,39.9,Nordeste
4,5,2025-03-10,Camiseta,Vestuário,8,29.9,Sudeste


#4. Uso do SELECT.

In [ ]:
query = '''
SELECT * FROM vendas LIMIT 5
'''
resultado = pd.read_sql_query(query, db)
print(resultado)

   id        data   produto   categoria  quantidade  preco_unitario    regiao
0   1  2025-01-15  Camiseta   Vestuário           5            29.9   Sudeste
1   2  2025-01-20     Calça   Vestuário           3            89.9       Sul
2   3  2025-02-05     Tênis    Calçados           2           199.9   Sudeste
3   4  2025-02-18      Boné  Acessórios          10            39.9  Nordeste
4   5  2025-03-10  Camiseta   Vestuário           8            29.9   Sudeste


#5. Usando WHERE e ORDER BY.

In [ ]:
query = '''
SELECT * FROM vendas
 WHERE quantidade >= 5
 ORDER BY quantidade
'''
resultado = pd.read_sql_query(query, db)
print(resultado)

   id        data   produto   categoria  quantidade  preco_unitario    regiao
0   1  2025-01-15  Camiseta   Vestuário           5            29.9   Sudeste
1  10  2025-05-20      Boné  Acessórios           6            39.9       Sul
2   5  2025-03-10  Camiseta   Vestuário           8            29.9   Sudeste
3   4  2025-02-18      Boné  Acessórios          10            39.9  Nordeste
4   9  2025-05-01  Camiseta   Vestuário          12            29.9   Sudeste
5   8  2025-04-14      Meia  Acessórios          15            19.9  Nordeste


#6. Escolhendo produtos distintos que começam com a letra C.

In [ ]:
query = '''
SELECT DISTINCT produto FROM vendas
 WHERE produto LIKE 'C%'
 ORDER BY produto
'''
resultado =pd.read_sql_query(query, db)
print(resultado)

    produto
0     Calça
1  Camiseta


#7. Valor médio por venda em cada região.

In [ ]:
query = '''
SELECT
    regiao,
    COUNT(*) as num_vendas,
    AVG(quantidade * preco_unitario) as ticket_medio
FROM vendas
GROUP BY regiao
ORDER BY ticket_medio DESC
'''

resultado = pd.read_sql_query(query, db)
print(resultado)


         regiao  num_vendas  ticket_medio
0       Sudeste           5    389.380000
1           Sul           3    369.600000
2  Centro-Oeste           1    359.600000
3      Nordeste           3    292.433333


# 8. Criação de uma nova tabela.

In [ ]:
#Tabela de categorias
db.execute('''
CREATE TABLE categorias (
  categoria TEXT PRIMARY KEY,
  setor TEXT,
  margem REAL
)
''')

db.executemany('INSERT INTO categorias VALUES (?, ?, ?)', [
    ('Vestuário', 'Moda', 0.35),
    ('Calçados', 'Moda', 0.40),
    ('Acessórios', 'Moda', 0.45)
])
db.commit()

print('Tabela de categorias criada com sucesso.')

Tabela de categorias criada com sucesso.


#8. Unir informações de margem por categoria.

In [ ]:
query = '''
SELECT
  categorias.categoria,
  categorias.setor,
  categorias.margem,
  SUM(vendas.quantidade*vendas.preco_unitario) AS fat_total
FROM categorias
INNER JOIN vendas ON categorias.categoria = vendas.categoria
GROUP BY categorias.categoria, categorias.setor,
categorias.margem
ORDER BY fat_total DESC
'''

resultado = pd.read_sql_query(query, db)
print(resultado)


    categoria setor  margem  fat_total
0    Calçados  Moda    0.40     1799.1
1   Vestuário  Moda    0.35     1556.6
2  Acessórios  Moda    0.45      936.9
